# Autodata — dezarhivare în cloud

Acest notebook lucrează în Google Colab. **Nu descarcă nimic pe telefon** și **nu publică arhiva**.

1. Alege **Runtime → Run all** și permite Colab să acceseze Google Drive.
2. Așteaptă finalizarea celulelor. Sursa din My Drive: **Autodata__3.45.rar**.
3. Rezultatele sunt salvate în **My Drive/Autodata_3_45_EXTRACTED/**, cu **INVENTAR.json**.

Verifică spațiul disponibil în Drive. Notebookul nu rulează niciun executabil extras; nu schimbă permisiuni de partajare.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montat.")

In [ ]:
import os, re, json, hashlib, shutil, subprocess, getpass
from pathlib import Path
from datetime import datetime, timezone

SOURCE = Path('/content/drive/MyDrive/Autodata__3.45.rar')
CACHE = Path('/content/autodata_work')
RAR = CACHE / 'Autodata__3.45.rar'
EXTRACT = CACHE / 'extract'
DEST = Path('/content/drive/MyDrive/Autodata_3_45_EXTRACTED')

assert SOURCE.is_file(), f'Arhiva nu a fost găsită: {SOURCE}'
CACHE.mkdir(parents=True, exist_ok=True)
if not shutil.which('7z'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'p7zip-full'], check=True)
size = SOURCE.stat().st_size
print(f'Arhiva: {SOURCE.name}, {size/2**30:.2f} GiB')
assert shutil.disk_usage('/content').free > size + 512*2**20, 'Spațiu insuficient în Colab'
digest = hashlib.sha256()
copied = 0
with SOURCE.open('rb') as src, RAR.open('wb') as dst:
    while True:
        block = src.read(8*2**20)
        if not block: break
        dst.write(block)
        digest.update(block)
        copied += len(block)
        if copied % (256*2**20) < len(block) or copied == size:
            print(f'Copiat în cloud: {copied/size:.0%}')
assert RAR.stat().st_size == size, 'Copie incompletă'
SHA256 = digest.hexdigest()
print('SHA256:', SHA256)

In [ ]:
def sevenzip(action, password=None):
    args = ['7z', action, '-y']
    if password:
        args.append('-p' + password)
    if action == 'x':
        args.append('-o' + str(EXTRACT))
    args.append(str(RAR))
    with (CACHE / f'{action}.log').open('w', encoding='utf-8') as fp:
        result = subprocess.run(args, stdin=subprocess.DEVNULL, stdout=fp,
                                stderr=subprocess.STDOUT, timeout=7200)
    return result.returncode

print('Verific integritatea arhivei...')
password = None
ok = sevenzip('t') == 0
if not ok:
    if input('Test eșuat. Este protejată cu parolă? (da/nu): ').strip().lower() in ('da','d','yes','y'):
        password = getpass.getpass('Parola arhivei: ')
        ok = sevenzip('t', password) == 0
if not ok:
    print((CACHE/'t.log').read_text(errors='replace')[-3000:])
    raise RuntimeError('Arhiva nu a trecut verificarea: nu o extrag.')

args = ['7z','l','-slt']
if password: args.append('-p'+password)
args.append(str(RAR))
listing = subprocess.run(args, capture_output=True, text=True, stdin=subprocess.DEVNULL,
                         timeout=1800, check=True, errors='replace').stdout
(CACHE/'listing.txt').write_text(listing, encoding='utf-8')
file_details = listing.split('----------',1)[-1]
expanded = sum(map(int, re.findall(r'(?m)^Size = (\d+)$', file_details)))
print('Dimensiune estimată după extragere:', round(expanded/2**30,2), 'GiB')
assert expanded <= 20*2**30, 'Arhiva s-ar extinde la peste 20 GiB: oprire preventivă.'
assert shutil.disk_usage('/content').free > expanded + 256*2**20, 'Spațiu local insuficient pentru extragere.'
EXTRACT.mkdir(exist_ok=True)
if sevenzip('x', password) != 0:
    print((CACHE/'x.log').read_text(errors='replace')[-3000:])
    raise RuntimeError('Extragerea a eșuat.')
print('Arhiva a fost extrasă în Colab.')

In [ ]:
records = []
for root, dirs, names in os.walk(EXTRACT, followlinks=False):
    root = Path(root)
    dirs[:] = [name for name in dirs if not (root/name).is_symlink()]
    for name in names:
        p = root/name
        if p.is_symlink() or not p.is_file():
            continue
        rel = p.relative_to(EXTRACT)
        records.append({'path': rel.as_posix(), 'bytes': p.stat().st_size})
print('Fișiere extrase:', len(records))
assert len(records) <= 120000, 'Prea multe fișiere; transferul a fost oprit.'
DEST.mkdir(parents=True, exist_ok=True)
for index, record in enumerate(records, 1):
    rel = Path(record['path'])
    src = EXTRACT / rel
    dst = DEST / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copyfile(src, dst)
    if index%100 == 0 or index == len(records):
        print(f'Salvat în Drive {index}/{len(records)}')

manifest = {'archive': SOURCE.name, 'archive_size': size, 'sha256': SHA256,
            'created_utc': datetime.now(timezone.utc).isoformat(),
            'files': records, 'count': len(records)}
(DEST/'INVENTAR.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
(DEST/'INVENTAR.txt').write_text(
    '\n'.join(f"{r['bytes']:>12}  {r['path']}" for r in records), encoding='utf-8')
for srcname, dstname in [('t.log','verificare_7z.txt'),('x.log','extragere_7z.txt'),
                         ('listing.txt','listare_7z.txt')]:
    shutil.copyfile(CACHE/srcname, DEST/dstname)
print('TERMINAT. My Drive / Autodata_3_45_EXTRACTED / INVENTAR.json')
print('Originalul a rămas intact; nu s-au executat programe din arhivă.')